# Module 5 · Structured Output & Function Calling

**Track:** LLMOps  
**Toolchain:** OpenAI Function Calling · Instructor · Outlines · Guidance  
**Objective:** Learn how to make LLMs return reliably structured data that your code can parse — bridging the gap between natural language and deterministic software.

---

## 🧠 The #1 Problem with LLMs in Production

LLMs return **free-form text.** Your code needs **structured data.** This mismatch causes 80% of production LLM bugs.

```python
# Module 5 · What you WANT from the LLM:
{"sentiment": "positive", "confidence": 0.92, "topics": ["pricing", "quality"]}

# Module 5 · What the LLM ACTUALLY returns:
"The sentiment of this review is mostly positive, with a confidence of around 92%.
 The main topics discussed are pricing and quality."

# Module 5 · Now try doing json.loads() on that 😱
```

### The Reliability Spectrum

| Method | Reliability | How It Works |
|--------|-----------|-------------|
| Ask nicely ("return JSON") | ~50% | Prompt-based. LLM often hallucinates markdown wrappers. |
| JSON mode (`response_format`) | ~95% | Forces valid syntax, but no schema guarantee. |
| Function Calling | ~99% | LLM fills API-defined argument schemas. |
| Instructor (Retries) | ~99.9% | Validates API output against Pydantic, auto-retries. |
| **Constrained Decoding** | **100%** | **Logit masking forces local open-source models to strictly follow a grammar.** |

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Parsing raw text outputs from an LLM (`json.loads(response)`) is the fastest way to crash a pipeline. Seniors FORCE the LLM to output guaranteed schemas. For APIs, they use prompt-engineering wrappers like `Instructor`. For open-source models deployed on local GPUs, they use strict **Constrained Decoding (Outlines/Guidance)** to guarantee mathematically perfect schemas at the token-generation level.

**Mental Model:** Function Calling via API is like asking an artist to fill in a strict blueprint. Constrained Decoding on a local GPU is like physically removing the artist's ability to draw anything except the lines on the blueprint.

---


## 📑 Table of Contents

* [1 · Function Calling: API Tools](#1--function-calling-api-tools)
* [2 · Pydantic + Instructor: API Reliability](#2--pydantic--instructor-api-reliability)
* [3 · Multi-Tool Orchestration](#3--multi-tool-orchestration)
* [4 · Deep Dive: Constrained Decoding (Local Models)](#4--deep-dive-constrained-decoding-local-models)
  * [How Logit Masking Works](#how-logit-masking-works)
  * [The Outlines Library](#the-outlines-library)
  * [The Guidance Library](#the-guidance-library)
* [✅ Knowledge Check](#-knowledge-check)
* [🔨 Practical Practice](#-practical-practice)
* [🎯 Summary & Key Takeaways](#-summary--key-takeaways)


---
## 1 · Function Calling: API Tools

Instead of asking the LLM to write JSON, you tell it: "Here are functions you can call. Pick the right one and fill in the arguments."

```
You define:    get_weather(city: str, unit: str = "celsius")
User says:     "What's the weather like in Paris?"
LLM returns:   {"function": "get_weather", "arguments": {"city": "Paris", "unit": "celsius"}}
Your code:     Actually calls get_weather("Paris", "celsius")
```


In [ ]:
# Cell 1 — Function calling flow simulation
import json
import random

tool_definitions = [
    {
        "type": "function",
        "function": {
            "name": "get_stock_price",
            "description": "Get the current stock price for a ticker symbol.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {"type": "string"},
                    "currency": {"type": "string", "enum": ["USD", "EUR"], "default": "USD"}
                },
                "required": ["ticker"]
            }
        }
    }
]

def simulate_function_calling(user_message, tools):
    if 'stock' in user_message.lower():
        return {'function': 'get_stock_price', 'arguments': {'ticker': 'AAPL', 'currency': 'USD'}}
    return None

print("LLM Output Intent:", simulate_function_calling("What's Apple stock?", tool_definitions))


---
## 2 · Pydantic + Instructor: API Reliability

Instructor wraps APIs (like OpenAI) and **guarantees** the output matches a Pydantic model by auto-retrying failures, injecting validation errors back into the prompt.

```
LLM → Pydantic validates → if invalid: retry(error msg) → Guaranteed Schema ✅
```


In [ ]:
# Cell 2 — Instructor Simulator
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional
import json

class UserInfo(BaseModel):
    name: str = Field(description="Full name of the user")
    age: int = Field(ge=0, description="Age in years")

class InstructorSimulator:
    def extract(self, raw_output):
        for attempt in range(3):
            try:
                return UserInfo(**json.loads(raw_output)), attempt+1
            except Exception as e:
                print(f"  ⚠️ Attempt {attempt+1} failed: Validation Error")
                raw_output = json.dumps({'name': 'Alice', 'age': 30}) # LLM auto-fixes
        raise ValueError("Max retries exceeded")

instructor = InstructorSimulator()
res, attempts = instructor.extract('{"name": "Alice", "age": "thirty"}')
print(f"\n✅ Final Payload: {res} (took {attempts} attempts)")


---
## 3 · Multi-Tool Orchestration

Production agents sequence multiple tools dynamically. The LLM acts as the orchestrator, planning a directed acyclic graph (DAG) of API calls.


In [ ]:
# Cell 3 — Simple Multi-Tool DAG Execution
print("Executing DAG:\n 1. get_stock(AAPL)\n 2. get_stock(GOOGL)\n 3. calc_diff(res1, res2)\n")
print("Final Diff: $23.20")


---
## 4 · Deep Dive: Constrained Decoding (Local Models)

When you own the model weights (e.g., deploying Llama-3 or Mistral via vLLM), you don't need prompt-engineering or retry-loops like `Instructor`. You can physically constrain the tokens the model is allowed to generate.

### How Logit Masking Works

LLMs predict the next word over a vocabulary of ~100,000 tokens. They output logits (probabilities) for every token.

In **Constrained Decoding** (FSM-driven output):
1. You compile a JSON Schema or Regex into a Finite State Machine (FSM).
2. At generation step $T$, the FSM calculates exactly which tokens are structurally legal.
3. The engine overrides the logits of all illegal tokens to $-\infty$.
4. The LLM is forced to pick the highest probability token *from the pool of syntactically legal tokens*.

**Result**: 100% Guaranteed JSON without retries, and significantly faster generation because the FSM essentially fast-forwards through deterministic tokens (like closing brackets `"}`).

### The Outlines Library
Outlines is the industry standard for regex-guided and JSON-guided generation on open-weights.


In [ ]:
# Cell 4 — Simulating Outlines Constrained Decoding
import re
import numpy as np

class MockOutlinesLogitProcessor:
    """
    Simulates how Outlines masks logits to enforce a Regex constraint.
    Goal Constraint: Must output a valid IP address.
    """
    def __init__(self):
        # Numpad tokens + periods
        self.vocab = {0: '1', 1: '9', 2: '2', 3: '.', 4: 'cat', 5: 'hello'}
        
    def process_logits(self, current_sequence, raw_logits):
        masked_logits = raw_logits.copy()
        
        # If we need a number next, mask out English words
        illegal_tokens = [4, 5]  # 'cat', 'hello'
        for idx in illegal_tokens:
            masked_logits[idx] = -np.inf
            
        return masked_logits

# Simulation
vocab_keys = [0, 1, 2, 3, 4, 5]
raw_llm_logits = np.array([2.0, 1.5, 3.2, 0.1, 8.9, 9.1]) # The LLM clearly wants to say 'hello' (9.1)

processor = MockOutlinesLogitProcessor()
masked = processor.process_logits("192.", raw_llm_logits)

print("🤖 Raw Logits (LLM Preference):")
for i, val in enumerate(raw_llm_logits):
    print(f"  {processor.vocab[i]:>5}: {val}")

print("\n🔒 Constrained Logits (Outlines Enforcement):")
for i, val in enumerate(masked):
    print(f"  {processor.vocab[i]:>5}: {val}")

best_token = processor.vocab[np.argmax(masked)]
print(f"\n✅ Chosen token: '{best_token}' (Because 'cat' and 'hello' were mathematically silenced)")


### The Guidance Library
Developed by Microsoft, Guidance allows interleaving generation with control logic directly in the prompt. It's powerful for complex agentic workflows where generation and python execution alternate rapidly within the same string.


---
## ✅ Knowledge Check

### Q1: API Wrappers vs Native Masks
Why can't you use `Outlines` (Logit Masking) to guarantee output from OpenAI integrations like GPT-4?
<details><summary>Answer</summary>
Logit masking requires manipulating the tensor probabilities inside the model's forward pass *before* the token is selected. OpenAI hides the forward pass behind an API and only returns text. Therefore, Logit Masking is strictly for open-weights models (via vLLM/LlamaCPP) where you control the GPU inference engine. For APIs, you must rely on retries (`Instructor`) or native endpoints (`Function Calling`).
</details>

---

## 🔨 Practical Practice

### Exercise 1: Outlines Regex
Install the `outlines` library and instantiate a HuggingFace pipeline. Create a regex pattern that forces a Llama model to output only completely valid JSON-lines containing an array of exactly three Hexadecimal color codes.


---
## 🎯 Summary & Key Takeaways

| Technique | Architecture | Mechanism | Best For |
|-----------|--------------|-----------|----------|
| Instructor | API Models | Validation + Dynamic Prompt Retry | Production OpenAI / Anthropic APIs |
| Function Calling | API Models | Fine-tuned Tool Argument Intent | Executing Web Searches, APIs |
| Outlines | Local Weights | Logit Masking + Pydantic/Regex FSM | High-throughput local generation |
| Guidance | Local Weights | Interleaved Programmatic Prompts | Complex control flows on local GPUs |

**Next →** `27_llmops_context_engineering.ipynb` — Module 6 — LLMOps & Context Engineering: The Agentic Frontier
